# Deepfake detector - Colab training

**Before running:** Runtime -> Change runtime type -> T4 GPU.

**One-time setup:** copy this project's `ml/` folder into your Google Drive (e.g. drag-and-drop
the folder into `My Drive/deepfake-image-detector/ml` via the Drive web UI). Everything in this
repo resolves paths relative to `ml/`'s own location, so once it's in Drive, data/checkpoints/
history all persist there automatically across sessions - no path changes needed.

**To resume in a later session:** just re-run every cell top to bottom. Manifest building is
skipped automatically if `manifest.csv` already exists, and training resumes from the last
checkpoint automatically (`configs/train.yaml` -> `checkpoint.resume: true`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU - go enable the T4 GPU runtime')
assert torch.cuda.is_available(), 'Enable a GPU runtime: Runtime -> Change runtime type -> T4 GPU'

In [ ]:
from pathlib import Path

# Update this if you placed the ml/ folder somewhere else in Drive.
PROJECT_DIR = Path('/content/drive/MyDrive/deepfake-image-detector/ml')
assert PROJECT_DIR.exists(), f'{PROJECT_DIR} not found - copy the ml/ folder into Drive first'
print('project dir ok:', PROJECT_DIR)

In [ ]:
# Alternative to manual upload, once you've pushed this repo to GitHub:
# !git clone https://github.com/<you>/deepfake-image-detector.git /content/drive/MyDrive/deepfake-image-detector
# PROJECT_DIR = Path('/content/drive/MyDrive/deepfake-image-detector/ml')

In [ ]:
!pip install -q -r {PROJECT_DIR}/requirements.txt

In [ ]:
import os
from google.colab import userdata

# Add HF_TOKEN under Colab's Secrets tab (key icon, left sidebar) first - never paste
# the raw token into a cell. Same free read-only token from huggingface.co/settings/tokens.
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_DIR))

from src.datasets.build_manifest import MANIFEST_PATH, build_manifest
from src.train import load_configs, train

In [ ]:
# First real validation run: ~800 images per source (~4-5k total across all 5 sources,
# split train/val/test + the genimage_pp held-out set). Skipped automatically on later
# sessions once manifest.csv already exists - won't re-download or re-sample.
PER_SOURCE_CAP = 800

if not MANIFEST_PATH.exists():
    rows = build_manifest(per_source_cap=PER_SOURCE_CAP)
    print(f'built manifest: {len(rows)} rows -> {MANIFEST_PATH}')
else:
    print(f'manifest already exists at {MANIFEST_PATH}, skipping build')

In [ ]:
model_cfg, train_cfg = load_configs()

result = train(model_cfg, train_cfg, checkpoint_dir=PROJECT_DIR / train_cfg['checkpoint']['dir'])
print('best val loss:', result['best_val_loss'])

Check progress anytime (even from another session) by opening, in Drive:
- `ml/checkpoints/training_curves.png` - train/val loss, per-branch loss, gate weights, accuracy
- `ml/checkpoints/canary/epoch_*.png` - explainability snapshots on the same fixed batch each epoch
- `ml/checkpoints/training_history.json` - raw per-epoch numbers